#Schema Enforcement:

When writing the data into table/file, ensure that data is in existing file/data format

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from pyspark.sql.functions import to_date

spark = SparkSession.builder.appName("MovieDeltaExample").getOrCreate()

data = [
    ("Inception", "2010-07-16", 8.8, "Sci-Fi"),
    ("Interstellar", "2014-11-07", 8.6, "Sci-Fi"),
    ("The Dark Knight", "2008-07-18", 9.0, "Action"),
    ("Tenet", "2020-08-26", 7.5, "Thriller"),
    ("Oppenheimer", "2023-07-21", 8.7, "Biography"),
    ("Dune", "2021-10-22", 8.0, "Sci-Fi"),
    ("Avatar", "2009-12-18", 7.8, "Fantasy"),
    ("The Matrix", "1999-03-31", 8.7, "Sci-Fi")
]

schema = StructType([
    StructField("title", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("genre", StringType(), True)
])

df = spark.createDataFrame(data, schema)
df = df.withColumn("release_date", to_date("release_date"))

df.show()


In [0]:
load_path="dbfs:/Volumes/databricks_practice/inputdb/moviesdata/movies_delta"
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(load_path)



In [0]:

spark.read.format("delta").load(load_path).display()

In [0]:
%sql
select * from delta.`dbfs:/Volumes/databricks_practice/inputdb/moviesdata/movies_delta` version as of 2

## Data Restortion 

In [0]:
spark.sql("RESTORE TABLE {TABLE_NAME} VERSION AS OF 1")

In [0]:
## for files:
# Read old version
load_path="dbfs:/Volumes/databricks_practice/inputdb/moviesdata/movies_delta"
old_df = spark.read.format("delta").option("versionAsOf", 1).load(load_path)

# Overwrite current table with old data
old_df.write.format("delta").mode("overwrite").save(load_path)

spark.read.format("delta").load(load_path).display()
